# Valuation Engine Prototype

This notebook serves as the experimental environment for the `ValuationEngine`. 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os
import xgboost as xgb
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_score, recall_score, make_scorer, mean_absolute_error, r2_score


# Add src to path if needed
sys.path.append(os.path.abspath("../src"))

#from valuation import ValuationEngine

%matplotlib inline
sns.set_theme(style="whitegrid")

## 1. Data Loading
We load the dataset processed by the `FeatureProcessor` (Gold Layer).

In [ ]:
data_path = "../data/processed/company_metrics_ml.parquet"

if os.path.exists(data_path):
    df = pd.read_parquet(data_path)
    print(f"Loaded {len(df)} companies with {len(df.columns)} columns.")
    display(df.head())
else:
    print(f"Data not found at {data_path}. Please run the pipeline first.")
    # Creating dummy data for demonstration if file doesn't exist
    df = pd.DataFrame({
        "forwardPE": np.random.uniform(10, 40, 100),
        "ev_to_ebitda": np.random.uniform(5, 20, 100),
        "ebitda_margin": np.random.uniform(0.1, 0.4, 100),
        "embeddings": [np.random.rand(384) for _ in range(100)]
    })
    display(df.head())

## 2. Model Training & Evaluation

In [ ]:
engine = ValuationEngine(model_type="random_forest")
X, y = engine.prepare_data(df, target_col="forwardPE")

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

engine.train(X_train, y_train)
metrics = engine.evaluate(X_test, y_test)

print("Performance Metrics:")
for k, v in metrics.items():
    print(f"{k.upper()}: {v:.4f}")

## 3. Visualization
Visualize the Predicted vs Actual valuations.

In [ ]:
y_pred = engine.predict(X_test)

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_test, y=y_pred, alpha=0.6)
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
plt.xlabel("Actual Forward PE")
plt.ylabel("Predicted Forward PE")
plt.title("Actual vs Predicted Valuation")
plt.show()